# Lab 1 — Simulate Potential Outcomes and Selection Bias
# المختبر 1 — محاكاة النواتج المحتملة وتحيّز الاختيار

**Module 1 · الوحدة 1 — Causal Thinking and Potential Outcomes / التفكير السببي والنواتج المحتملة**
SDA-DSC-213 · Experimentation, A/B Testing and Causal Inference · SDAIA Academy

---

### The decision on the table

Injaz (إنجاز) is piloting a **guided document uploader**. During the soft launch,
users could *opt in*. The dashboard says opt-in users complete services **19 points**
more often than everyone else. A director wants to spend the national rollout budget
on that number.

Your job in the next 50 minutes is to prove — in a world where you can *see* the
right answer — that this 19 points is mostly **selection bias (تحيّز الاختيار)**, and
that a coin flip would have given the honest number.

### What you will build

| # | Step | Minutes |
|---|---|---|
| 1 | A simulator with **both** potential outcomes and a planted ground-truth effect | 8 |
| 2 | Two assignment mechanisms — self-selection and randomisation — plus the reveal rule | 10 |
| 3 | The naive difference-in-means under each mechanism | 10 |
| 4 | The selection-bias term computed *directly*, and the identity naive = ATT + bias | 10 |
| 5 | The sample-size sweep that kills the "we have millions of rows" defence | 7 |
| 6 | A three-sentence conclusion a non-technical director can read | 5 |

### The vocabulary this lab installs (bilingual, once)

- **potential outcomes (النواتج المحتملة)** — Y(1) and Y(0): the two outcomes the same
  unit would have with and without treatment.
- **counterfactual (النتيجة المضادة للواقع)** — the one of those two you did *not* observe.
- **ATE (متوسط أثر المعالجة)** — E[Y(1) − Y(0)] over everyone.
- **ATT (متوسط أثر المعالجة على المُعالَجين)** — E[Y(1) − Y(0) | T = 1], the effect among
  those who actually got it.
- **selection bias (تحيّز الاختيار)** — E[Y(0)|T=1] − E[Y(0)|T=0]: how different the two
  groups would have been *with no treatment at all*.
- **confounder (المتغيّر المُربِك)** — here, digital literacy: it raises both opt-in and completion.

> All code, variables and comments stay in **English**. Only the concepts are bilingual.


---
## Step 0 — Environment and the numbers we are aiming at

`causal_utils` is the course toolkit you are assembling across Labs 1–7. Lab 1
contributes the **simulator and the bias decomposition** — the ground-truth harness
every later method gets checked against.

`GROUND_TRUTH.json` holds the numbers that were *deliberately planted* in the course
datasets. Read it now, so that at the end of the lab you are comparing against a
target you did not choose after the fact.


In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# Make `causal_utils` importable from anywhere under course_assets/
ROOT = Path.cwd()
while not (ROOT / "causal_utils").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
DATA = ROOT / "data"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from causal_utils import use_sdaia_style
use_sdaia_style()

RANDOM_SEED = 213          # fixed for every lab: reproducibility is graded
rng = np.random.default_rng(RANDOM_SEED)
print("Environment ready ·", ROOT)

In [ ]:
def needs(*values, msg="complete the TODO in the cell above, then re-run this cell"):
    """Guard for the start notebook: True (with a friendly note) if a TODO is unfinished.

    This is why the start notebook runs top-to-bottom without a traceback even
    before you have written a line of code.
    """
    if any(v is None for v in values):
        print(f"[not implemented yet] {msg}")
        return True
    return False


def sigmoid(x):
    """Logistic function — turns a real-valued index into a probability."""
    return 1.0 / (1.0 + np.exp(-x))

In [ ]:
import json

with open(DATA / "GROUND_TRUTH.json") as f:
    GROUND_TRUTH = json.load(f)

M1 = GROUND_TRUTH["module1_potential_outcomes"]
print("Planted ground truth for Module 1")
print(f"  true ATE                    : {M1['true_ATE']:+.4f}   ({100*M1['true_ATE']:+.2f} pp)")
print(f"  naive, self-selected groups : {M1['naive_self_selected']:+.4f}   ({100*M1['naive_self_selected']:+.2f} pp)")
print(f"  naive, randomised groups    : {M1['naive_randomised']:+.4f}   ({100*M1['naive_randomised']:+.2f} pp)")
print(f"  selection-bias term         : {M1['selection_bias_term']:+.4f}   ({100*M1['selection_bias_term']:+.2f} pp)")

---
## Step 1 *(8 min)* — Build the simulator

The whole course rests on one teaching device: **plant an effect you can see, then
check which estimator recovers it.** In reality Y(1) and Y(0) never both exist for
the same person — that is the *fundamental problem of causal inference*. In a
simulator they do, which is the only place you can grade an estimator honestly.

The data-generating process, deliberately simple:

```
literacy_i  ~ Beta(2, 2)                     # latent digital literacy, mean 0.5
p0_i        = 0.55 + beta_literacy * (literacy_i - 0.5) + 0.09 * user_effect_i
p1_i        = p0_i + true_effect             # the treatment adds a constant on the probability scale
y0_i        = 1{u_i < p0_i},  y1_i = 1{u_i < p1_i}     # SAME u_i -> coupled potential outcomes
p_optin_i   = sigmoid(k_select * (literacy_i - 0.5))   # literacy drives who opts in
```

Two design choices worth pausing on:

1. **`literacy` enters both `p0` and `p_optin`.** That is exactly what makes it a
   *confounder*: it lifts completion whether or not you treat, and it lifts adoption.
2. **`y0` and `y1` share the same uniform draw `u_i`.** Without that coupling the
   individual effect `tau_i` would be mostly draw noise. Coupling gives a clean
   monotone treatment: nobody is hurt by the uploader.


In [ ]:
def simulate_injaz_users(n=20_000, true_effect=0.05, beta_literacy=1.3284,
                         k_select=2.5, seed=RANDOM_SEED):
    """TODO: simulate Injaz users where BOTH potential outcomes are known.

    Implement, in this order (the rng call order matters for reproducibility):

      1. literacy    = g.beta(2, 2, size=n);  lit_c = literacy - 0.5
      2. user_effect = g.normal(0, 1, size=n)
      3. p0 = np.clip(0.55 + beta_literacy * lit_c + 0.09 * user_effect, 0.04, 0.95)
      4. p1 = np.clip(p0 + true_effect, 0.04, 0.99)
      5. u  = g.random(n);  y0 = (u < p0).astype(int);  y1 = (u < p1).astype(int)
         -> the SAME u for both: coupled potential outcomes
      6. p_optin = sigmoid(k_select * lit_c)

    Return a DataFrame with columns:
      digital_literacy, user_effect, p0, p1, y0, y1, tau_i (= y1 - y0), p_optin

    Return None while unfinished so the notebook still runs end to end.
    """
    g = np.random.default_rng(seed)

    # TODO: build the six pieces above and return the DataFrame
    return None


users = simulate_injaz_users()

if needs(users, msg="implement simulate_injaz_users"):
    true_ate = None
else:
    true_ate = users["tau_i"].mean()
    assert 0.040 < true_ate < 0.055, f"expected the planted ATE near +0.05, got {true_ate:.4f}"
    assert set(["y0", "y1", "tau_i", "p_optin"]).issubset(users.columns), "missing columns"
    print(f"n                          : {len(users):,}")
    print(f"True ATE (planted, realised): {true_ate:+.4f}  ({100*true_ate:+.2f} pp)")
    print(f"Baseline completion  E[Y(0)]: {users['y0'].mean():.4f}")
    lo, hi = users["digital_literacy"].quantile([0.1, 0.9])
    print(f"P(opt in) bottom-decile literacy: "
          f"{users.loc[users.digital_literacy <= lo, 'p_optin'].mean():.3f}")
    print(f"P(opt in) top-decile literacy   : "
          f"{users.loc[users.digital_literacy >= hi, 'p_optin'].mean():.3f}")
    print("OK — the planted ATE is back. You now have a world where the truth is visible.")

---
## Step 2 *(10 min)* — Two assignment mechanisms, one reveal rule

Same users. Same true effect. **Only the assignment mechanism differs.**

- **Self-selection:** `T_self ~ Bernoulli(p_optin)` — literate users opt in more.
- **Randomisation:** `T_rand ~ Bernoulli(0.5)` — a coin flip, independent of everything.

Then the *consistency* rule links potential outcomes to what you actually observe:

$$Y_i = T_i\,Y_i(1) + (1 - T_i)\,Y_i(0)$$

In code that is `np.where(T == 1, y1, y0)`. Get this backwards and your estimate
flips sign — which is exactly the debugging exercise at the end of this lab.


In [ ]:
def assign_self_selection(df, seed=RANDOM_SEED + 1):
    """TODO: opt-in assignment. Return a 0/1 array where P(T=1) = df["p_optin"].

    Hint: g = np.random.default_rng(seed); (g.random(len(df)) < p_optin).astype(int)
    """
    return None


def assign_randomised(df, share=0.5, seed=RANDOM_SEED + 2):
    """TODO: coin-flip assignment. Return a 0/1 array where P(T=1) = share,
    independent of every column in df."""
    return None


def reveal_observed(df, t_col):
    """TODO: apply consistency,  Y = T*Y(1) + (1-T)*Y(0).

    Return an array: y1 where the unit is treated, y0 where it is not.
    Getting this backwards is the single most common bug in this lab.
    """
    return None


t_self = assign_self_selection(users) if users is not None else None
t_rand = assign_randomised(users) if users is not None else None

if not needs(users, t_self, t_rand, msg="implement the two assignment functions"):
    users["T_self"] = t_self
    users["T_rand"] = t_rand
    y_self = reveal_observed(users, "T_self")
    y_rand = reveal_observed(users, "T_rand")
    if not needs(y_self, y_rand, msg="implement reveal_observed"):
        users["y_self"] = y_self
        users["y_rand"] = y_rand
        assert abs(users["T_rand"].mean() - 0.5) < 0.02, "randomised share should be ~0.5"
        lit_gap = (users.loc[users.T_self == 1, "digital_literacy"].mean()
                   - users.loc[users.T_self == 0, "digital_literacy"].mean())
        assert lit_gap > 0.05, ("self-selected treated users must be MORE literate; "
                                f"gap is only {lit_gap:.3f} — check p_optin")
        print(f"treated share, self-selection : {users['T_self'].mean():.3f}")
        print(f"treated share, randomisation  : {users['T_rand'].mean():.3f}")
        print(f"literacy gap (self-selection) : {lit_gap:+.3f}   <- the confounding, visible")
        print("OK — assignment and reveal are in place.")

### ✅ Checkpoint A — what you should be seeing by now

Run the cell below. If it prints all three ticks you are on pace (≈18 minutes in).


In [ ]:
def checkpoint_a():
    ok = True
    if users is None or "y_self" not in users.columns:
        print("✗ Steps 1-2 incomplete — the simulator or the reveal rule is still a TODO.")
        return
    print(f"✓ simulator built           : {len(users):,} users, both Y(0) and Y(1) present")
    print(f"✓ planted ATE recovered     : {users['tau_i'].mean():+.4f}  (target ≈ +0.05)")
    lit_gap = (users.loc[users.T_self == 1, "digital_literacy"].mean()
               - users.loc[users.T_self == 0, "digital_literacy"].mean())
    print(f"✓ confounding is present    : literacy gap {lit_gap:+.3f} under self-selection, "
          f"{users.loc[users.T_rand == 1, 'digital_literacy'].mean() - users.loc[users.T_rand == 0, 'digital_literacy'].mean():+.3f} under randomisation")
    print()
    print("Next: the naive difference under each mechanism, and why they disagree.")

checkpoint_a()

---
## Step 3 *(10 min)* — The naive difference under each mechanism

The estimator every dashboard in the world computes:

$$\hat\tau_{\text{naive}} = \bar{Y}_{T=1} - \bar{Y}_{T=0}$$

Run it on both mechanisms. The true effect is **identical** in the two columns — the
only thing that changed is *who* got treated.


In [ ]:
def naive_diff(df, t_col, y_col):
    """TODO: return mean(y | T=1) - mean(y | T=0) as a float.

    assert-able target: on this simulator, self-selection lands near +0.19
    and randomisation near +0.05.
    """
    return None


est_self = naive_diff(users, "T_self", "y_self") if "y_self" in (users.columns if users is not None else []) else None
est_rand = naive_diff(users, "T_rand", "y_rand") if "y_rand" in (users.columns if users is not None else []) else None

if not needs(est_self, est_rand, msg="implement naive_diff"):
    assert 0.15 < est_self < 0.24, f"self-selected estimate should be ~+0.19, got {est_self:+.4f}"
    assert 0.03 < est_rand < 0.08, f"randomised estimate should be ~+0.05, got {est_rand:+.4f}"
    print(f"True ATE (we can see it)   : {true_ate:+.4f}   ({100*true_ate:+.2f} pp)")
    print(f"Naive, self-selected groups: {est_self:+.4f}   ({100*est_self:+.2f} pp)  <- biased HIGH")
    print(f"Naive, randomised groups   : {est_rand:+.4f}   ({100*est_rand:+.2f} pp)  <- ~unbiased")
    print(f"\nSelf-selection overstates the effect by a factor of {est_self / true_ate:.1f}x.")

**Write your answer here before moving on.** Two sentences, in your own words:

*Why do these two numbers differ when the true effect is identical in both columns?*

> _(your answer)_


---
## Step 4 *(10 min)* — Decompose the naive difference

The algebra derived on the whiteboard:

$$\underbrace{E[Y|T{=}1] - E[Y|T{=}0]}_{\text{naive}} \;=\; \underbrace{E[Y(1)-Y(0)\mid T{=}1]}_{\textbf{ATT}} \;+\; \underbrace{E[Y(0)\mid T{=}1] - E[Y(0)\mid T{=}0]}_{\textbf{selection bias}}$$

In real data the second term is **unmeasurable** — you never see Y(0) for treated
units. In this simulator you do. Compute it directly and confirm the identity holds
to the fourth decimal.


In [ ]:
def decompose(df, t_col, y_col):
    """TODO: split the naive difference into ATT + selection bias.

    Return a dict with keys:
      mechanism, naive, ATT, selection_bias, ATT_plus_bias, identity_gap

      ATT            = mean of tau_i among treated units
      selection_bias = mean(y0 | T=1) - mean(y0 | T=0)      <- uses the counterfactual!
      identity_gap   = naive - (ATT + selection_bias)       <- must be ~0
    """
    return None


_rows = [decompose(users, "T_self", "y_self"), decompose(users, "T_rand", "y_rand")]

if not needs(*_rows, msg="implement decompose"):
    decomp = pd.DataFrame(_rows)
    assert abs(decomp["identity_gap"]).max() < 1e-9, \
        "naive must equal ATT + selection_bias exactly — check your indexing"
    assert decomp.loc[0, "selection_bias"] > 0.10, \
        "self-selection bias should be large and positive (~+0.14)"
    assert abs(decomp.loc[1, "selection_bias"]) < 0.03, \
        "randomised bias should be ~0"
    print(decomp.round(4).to_string(index=False))
    print("\nThe identity holds exactly: naive = ATT + selection bias, by construction.")
    print(f"Under self-selection the bias term is "
          f"{100*decomp.loc[0, 'selection_bias']:+.2f} pp — "
          f"{100*decomp.loc[0, 'selection_bias'] / decomp.loc[0, 'naive']:.0f}% of the headline.")
else:
    decomp = None

In [ ]:
from causal_utils import NAVY, BLUE, ORANGE, TEAL, SLATE, MUTED

def plot_decomposition(decomp):
    fig, ax = plt.subplots(figsize=(9, 4.4))
    labels = ["Self-selected\n(opt-in soft launch)", "Randomised\n(coin flip)"]
    att = 100 * decomp["ATT"].to_numpy()
    bias = 100 * decomp["selection_bias"].to_numpy()

    ax.barh(labels, att, color=TEAL, label="ATT — the real effect", height=0.5)
    ax.barh(labels, bias, left=att, color=ORANGE, label="selection bias", height=0.5)
    ax.axvline(100 * true_ate, color=NAVY, ls="--", lw=1.6,
               label=f"true ATE ({100*true_ate:+.2f} pp)")

    for i, (a, b) in enumerate(zip(att, bias)):
        ax.text(a + b + 0.4, i, f"naive = {a + b:+.2f} pp", va="center",
                fontsize=10, color=NAVY, fontweight="bold")

    ax.set_xlabel("Effect on service-completion rate (percentage points)")
    ax.set_title("The naive difference is ATT plus selection bias")
    ax.legend(loc="lower right")
    ax.set_xlim(-1, 100 * (att + bias).max() * 1.45)
    ax.annotate(
        "Takeaway: randomisation does not make the effect bigger.\n"
        "It removes the orange block — the part that was never the uploader.",
        xy=(0.02, 0.06), xycoords="axes fraction", fontsize=10, color=SLATE,
        bbox=dict(boxstyle="round,pad=0.5", fc="#F4F7FB", ec=MUTED, lw=0.8))
    plt.tight_layout()
    plt.show()


if not needs(decomp, msg="implement decompose in Step 4"):
    plot_decomposition(decomp)

### ✅ Checkpoint B — what you should be seeing by now

≈38 minutes in. You should have a table where the self-selection row reads roughly
`naive +0.19 = ATT +0.05 + bias +0.14` and the randomised row reads
`naive +0.05 = ATT +0.05 + bias +0.00`.


In [ ]:
def checkpoint_b():
    if decomp is None:
        print("✗ Step 4 incomplete — the decomposition is still a TODO.")
        return
    s = decomp.set_index("mechanism")
    print(f"✓ naive, self-selected : {s.loc['T_self', 'naive']:+.4f}   (target ≈ +0.19)")
    print(f"✓ naive, randomised    : {s.loc['T_rand', 'naive']:+.4f}   (target ≈ +0.05)")
    print(f"✓ bias, self-selected  : {s.loc['T_self', 'selection_bias']:+.4f}   (target ≈ +0.14)")
    print(f"✓ bias, randomised     : {s.loc['T_rand', 'selection_bias']:+.4f}   (target ≈  0.00)")
    print("\nNext: the sample-size sweep — the part that changes how you argue with stakeholders.")

checkpoint_b()

---
## Step 5 *(7 min)* — Bias does not shrink with data

The most expensive misconception in applied data science: *"we have four million rows,
so the result must be reliable."*

More data shrinks **variance**. It does nothing at all to **bias**. Sweep
n ∈ {1k, 10k, 100k, 1M}, five independent replicates each, and watch the two curves
converge — to two different numbers.


In [ ]:
def sweep_sample_size(sizes=(1_000, 10_000, 100_000, 1_000_000), n_reps=5):
    """TODO: re-run the simulation at each sample size, n_reps times.

    For each (n, rep):
      - d = simulate_injaz_users(n=n, seed=1_000 + rep)
      - assign both mechanisms (use seeds 2_000 + rep and 3_000 + rep)
      - reveal both observed outcome columns
      - record n, rep, self_selected estimate, randomised estimate, true_ate

    Return a tidy DataFrame with those five columns.
    """
    return None


sweep = sweep_sample_size()

if not needs(sweep, msg="implement sweep_sample_size"):
    summary = sweep.groupby("n")[["self_selected", "randomised", "true_ate"]].agg(["mean", "std"])
    big = summary.loc[1_000_000]
    assert abs(big[("randomised", "mean")] - big[("true_ate", "mean")]) < 0.005, \
        "at n = 1M the randomised estimate must sit on the truth"
    assert big[("self_selected", "mean")] > 0.15, \
        "at n = 1M the self-selected estimate must still be ~+0.19 — bias does not shrink"
    print(summary.round(4).to_string())
else:
    summary = None

In [ ]:
def plot_convergence(sweep):
    agg = sweep.groupby("n").agg(
        self_mean=("self_selected", "mean"), self_sd=("self_selected", "std"),
        rand_mean=("randomised", "mean"), rand_sd=("randomised", "std"),
        truth=("true_ate", "mean")).reset_index()

    fig, ax = plt.subplots(figsize=(9.5, 5))
    x = agg["n"]
    for col_m, col_s, colour, label in [
            ("self_mean", "self_sd", ORANGE, "Self-selected (opt-in) comparison"),
            ("rand_mean", "rand_sd", BLUE, "Randomised comparison")]:
        ax.plot(x, 100 * agg[col_m], "o-", color=colour, lw=2.2, ms=7, label=label)
        ax.fill_between(x, 100 * (agg[col_m] - agg[col_s]), 100 * (agg[col_m] + agg[col_s]),
                        color=colour, alpha=0.15)
    ax.axhline(100 * agg["truth"].mean(), color=NAVY, ls="--", lw=1.6,
               label=f"true ATE ({100*agg['truth'].mean():+.2f} pp)")

    ax.set_xscale("log")
    ax.set_xlabel("Sample size (log scale)")
    ax.set_ylabel("Estimated effect on completion rate (percentage points)")
    ax.set_title("More data does not fix a biased comparison")
    ax.legend(loc="center right")

    gap = 100 * (agg["self_mean"].iloc[-1] - agg["truth"].mean())
    ax.annotate("", xy=(1.05e6, 100 * agg["self_mean"].iloc[-1]),
                xytext=(1.05e6, 100 * agg["truth"].mean()),
                arrowprops=dict(arrowstyle="<->", color=SLATE, lw=1.4))
    ax.text(1.15e6, 100 * (agg["self_mean"].iloc[-1] + agg["truth"].mean()) / 2,
            f"gap = {gap:+.1f} pp\nand it is NOT closing", va="center",
            fontsize=10, color=SLATE)
    ax.set_xlim(700, 4e6)
    ax.annotate(
        "Takeaway: the bands narrow (variance falls) but the orange line does not move\n"
        "toward the truth. n buys precision, never identification.",
        xy=(0.02, 0.06), xycoords="axes fraction", fontsize=10, color=SLATE,
        bbox=dict(boxstyle="round,pad=0.5", fc="#F4F7FB", ec=MUTED, lw=0.8))
    plt.tight_layout()
    plt.show()


if not needs(sweep, msg="implement sweep_sample_size in Step 5"):
    plot_convergence(sweep)

---
## Step 6 — Two extras that pay off later

**(a) The confounding dial.** Bias magnitude is set by *confounding strength*, not by n.
Sweep `k_select` (how strongly literacy drives opt-in) and watch only the orange line move.

**(b) Adjusting for the confounder.** If you *measured* the confounder you can regress it
away. Here literacy is fully observed, so adjustment nearly recovers the truth — which is
the whole promise, and the whole fragility, of Modules 5 and 6: it works only for the
confounders you named and measured.


In [ ]:
# (a) confounding dial — TODO: fill the loop below
dial = None
# TODO: for k in [0.0, 1.0, 2.0, 2.5, 4.0, 6.0]:
#           simulate at n=100_000 with k_select=k, assign both mechanisms,
#           record the two naive estimates. Build a DataFrame called `dial`.

# (b) covariate adjustment — TODO: regress the observed outcome on T_self,
#     then on T_self + digital_literacy, and compare the coefficients.
adjusted_estimate = None

if not needs(dial, msg="fill the confounding-dial loop"):
    print(dial.round(4).to_string(index=False))
if not needs(adjusted_estimate, msg="run the covariate-adjusted regression"):
    print(f"adjusted estimate: {adjusted_estimate:+.4f}   (true ATE {true_ate:+.4f})")

---
## Step 7 — Grade yourself against the planted ground truth

Your simulator is one realisation of the course's Module-1 data-generating process.
The file `simulated_potential_outcomes.parquet` is another — 20,000 Injaz users with
`y0`, `y1`, `tau_individual` and both assignment mechanisms already recorded. It is the
**only** file in the entire course that contains both potential outcomes.

Run the same three estimators on it and compare every number to `GROUND_TRUTH.json`.


In [ ]:
po = pd.read_parquet(DATA / "simulated_potential_outcomes.parquet")


def grade_against_ground_truth():
    if needs(true_ate, est_self, est_rand, msg="finish Steps 1-3, then re-run"):
        return
    planted = {
        "true ATE": (po["tau_individual"].mean(), M1["true_ATE"]),
        "naive, self-selected": (
            naive_diff(po, "treated_self_selected", "y_observed_self_selected"),
            M1["naive_self_selected"]),
        "naive, randomised": (
            naive_diff(po, "treated_randomised", "y_observed_randomised"),
            M1["naive_randomised"]),
        "selection-bias term": (
            float(po.loc[po.treated_self_selected == 1, "y0"].mean()
                  - po.loc[po.treated_self_selected == 0, "y0"].mean()),
            M1["selection_bias_term"]),
    }
    print(f"{'quantity':<24}{'recovered':>12}{'planted':>12}{'gap':>10}   verdict")
    print("-" * 72)
    for name, (got, want) in planted.items():
        gap = got - want
        print(f"{name:<24}{got:>+12.4f}{want:>+12.4f}{gap:>+10.4f}   "
              f"{'MATCH' if abs(gap) < 0.002 else 'CHECK'}")
    print("\nAnd your own simulator, for comparison:")
    print(f"{'true ATE (yours)':<24}{true_ate:>+12.4f}{M1['true_ATE']:>+12.4f}")
    print(f"{'naive self (yours)':<24}{est_self:>+12.4f}{M1['naive_self_selected']:>+12.4f}")
    print(f"{'naive rand (yours)':<24}{est_rand:>+12.4f}{M1['naive_randomised']:>+12.4f}")
    print("\nBenchmark for Module 1: the randomised estimate must land within +/- 0.005 of")
    print("the planted ATE, and the selection-bias term must reproduce to three decimals.")


grade_against_ground_truth()

---
## Step 8 *(5 min)* — The three sentences a director reads

Write your conclusion in the cell below. Constraints: **three sentences, no jargon, no
Greek letters**, and it must state a number, a caveat, and a recommended action.


In [ ]:
memo = """
TODO: three sentences. State (1) what the 19 points actually is, (2) what the honest
number is and how you know, (3) what you recommend the director do next.
"""
print(memo)

---
## What you now own

Lab 1 contributes the **ground-truth harness** to your experimentation toolkit. Every
method in Modules 2–7 gets pointed at a simulator like this one *before* it is pointed
at data where the answer is unknown.


In [ ]:
print("""
TOOLKIT CONTRIBUTION — Lab 1
============================================================
Reusable artefacts you built and can lift into any project:

  simulate_injaz_users(n, true_effect, beta_literacy, k_select, seed)
      A potential-outcomes generator with a PLANTED effect and a tunable
      confounder. Use it to grade any new estimator before trusting it.

  reveal_observed(df, t_col)
      The consistency rule, Y = T*Y(1) + (1-T)*Y(0). One line, and the
      source of the most common sign error in causal code.

  naive_diff(df, t_col, y_col)
      The estimator every dashboard computes. You now know exactly what
      it estimates: ATT + selection bias.

  decompose(df, t_col, y_col)
      naive = ATT + selection bias, computed directly. The diagnostic that
      tells you HOW WRONG a naive comparison is, whenever you can simulate.

  sweep_sample_size(...)
      The bias-vs-n exhibit. Keep the plot; you will need it the next time
      someone says "but we have millions of rows".

Carried forward:
  - Lab 2 randomises for real, so the bias term is zero BY DESIGN.
  - Labs 5-6 return to observational data, where the bias term is back and
    must be argued away with assumptions instead of a coin flip.
============================================================
""")

---
## Mini-exercises

### 1 · Quiz (answer in this cell, then check against Module 1)

1. What are the two potential outcomes for a unit under a binary treatment?
2. State the fundamental problem of causal inference in one sentence.
3. Decompose the naive difference of means.
4. Why does randomisation eliminate selection bias?
5. True or false: with enough data, selection bias disappears.

### 2 · Debugging exercise — the inverted counterfactual

The cell below reveals the observed outcome as `np.where(T == 1, y0, y1)` — the two
potential outcomes are **swapped**. Run it, look at the sign, and state in one sentence
what a stakeholder would have concluded from this notebook.

### 3 · Estimand exercise

For each Injaz decision, write the estimand (**ATE / ATT / CATE**) and the
potential-outcomes contrast in one line:

| Decision | Estimand | Contrast |
|---|---|---|
| "Should we launch the guided uploader to all users?" | | |
| "Did the digital-skills workshop help the citizens who attended?" | | |
| "What is the uploader's effect for users aged 60+?" | | |

### 4 · Discussion

- A stakeholder says *"we have 4 million rows, the result must be reliable."* What do
  you say — in one sentence, without using the word "bias"?
- When is **ATT** the more relevant estimand than **ATE** for an Injaz programme?


In [ ]:
# --- Mini-exercise 2: the inverted counterfactual -------------------------
def reveal_observed_BROKEN(df, t_col):
    """DELIBERATELY WRONG: the potential outcomes are swapped."""
    return np.where(df[t_col].to_numpy() == 1, df["y0"].to_numpy(), df["y1"].to_numpy())


if not needs(users, est_self, est_rand, msg="finish Steps 1-3 first"):
    broken = users.copy()
    broken["y_broken_self"] = reveal_observed_BROKEN(broken, "T_self")
    broken["y_broken_rand"] = reveal_observed_BROKEN(broken, "T_rand")
    print(f"{'':<34}{'correct':>10}{'swapped':>10}")
    print("-" * 54)
    print(f"{'randomised assignment':<34}{est_rand:>+10.4f}"
          f"{naive_diff(broken, 'T_rand', 'y_broken_rand'):>+10.4f}")
    print(f"{'self-selected assignment':<34}{est_self:>+10.4f}"
          f"{naive_diff(broken, 'T_self', 'y_broken_self'):>+10.4f}")
    print("\nWrite one sentence for EACH row: what would a stakeholder have concluded?")